In [1]:
# Import PuLP modeler functions
from pulp import *

In [3]:
# Creates a list of the Ingredients
Ingredients = ['CHICKEN', 'BEEF', 'MUTTON', 'RICE', 'WHEAT', 'GEL']
# A dictionary of the costs of each of the Ingredients is created
costs = {'CHICKEN': 0.013,
         'BEEF': 0.008,
         'MUTTON': 0.010,
         'RICE': 0.002,
         'WHEAT': 0.005,
         'GEL': 0.001}
# A dictionary of the protein percent in each of the Ingredients is created
proteinPercent = {'CHICKEN': 0.100,
                  'BEEF': 0.200,
                  'MUTTON': 0.150,
                  'RICE': 0.000,
                  'WHEAT': 0.040,
                  'GEL': 0.000}
# A dictionary of the fat percent in each of the Ingredients is created
fatPercent = {'CHICKEN': 0.080,
              'BEEF': 0.100,
              'MUTTON': 0.110,
              'RICE': 0.010,
              'WHEAT': 0.010,
              'GEL': 0.000}
# A dictionary of the fibre percent in each of the Ingredients is created
fibrePercent = {'CHICKEN': 0.001,
                'BEEF': 0.005,
                'MUTTON': 0.003,
                'RICE': 0.100,
                'WHEAT': 0.150,
                'GEL': 0.000}
# A dictionary of the salt percent in each of the Ingredients is created
saltPercent = {'CHICKEN': 0.002,
               'BEEF': 0.005,
               'MUTTON': 0.007,
               'RICE': 0.002,
               'WHEAT': 0.008,
               'GEL': 0.000}

In [5]:
# Create the 'prob' variable to contain the problem data
prob = LpProblem("Blending_Model", LpMinimize)
# A dictionary called 'ingredient_vars' is created to contain the referenced Variables
ingredient_vars = LpVariable.dicts("Ingr",Ingredients,0)
#The objective function is added to 'prob' first

In [7]:
ingredient_vars

{'CHICKEN': Ingr_CHICKEN,
 'BEEF': Ingr_BEEF,
 'MUTTON': Ingr_MUTTON,
 'RICE': Ingr_RICE,
 'WHEAT': Ingr_WHEAT,
 'GEL': Ingr_GEL}

In [9]:
#The objective function is added to 'prob' first
prob += lpSum([costs[i]*ingredient_vars[i] for i in Ingredients]), "Total Cost of Ingredients per 100 gr can"

In [11]:
# The five constraints are added to 'prob'
prob += lpSum([ingredient_vars[i] for i in Ingredients]) == 100, "GramsSum"
prob += lpSum([proteinPercent[i] * ingredient_vars[i] for i in Ingredients]) >= 8.0, "ProteinRequirement"
prob += lpSum([fatPercent[i] * ingredient_vars[i] for i in Ingredients]) >= 6.0, "FatRequirement"
prob += lpSum([fibrePercent[i] * ingredient_vars[i] for i in Ingredients]) <= 2.0, "FibreRequirement"
prob += lpSum([saltPercent[i] * ingredient_vars[i] for i in Ingredients]) <= 0.4, "SaltRequirement"

In [13]:
prob

Blending_Model:
MINIMIZE
0.008*Ingr_BEEF + 0.013*Ingr_CHICKEN + 0.001*Ingr_GEL + 0.01*Ingr_MUTTON + 0.002*Ingr_RICE + 0.005*Ingr_WHEAT + 0.0
SUBJECT TO
GramsSum: Ingr_BEEF + Ingr_CHICKEN + Ingr_GEL + Ingr_MUTTON + Ingr_RICE
 + Ingr_WHEAT = 100

ProteinRequirement: 0.2 Ingr_BEEF + 0.1 Ingr_CHICKEN + 0.15 Ingr_MUTTON
 + 0.04 Ingr_WHEAT >= 8

FatRequirement: 0.1 Ingr_BEEF + 0.08 Ingr_CHICKEN + 0.11 Ingr_MUTTON
 + 0.01 Ingr_RICE + 0.01 Ingr_WHEAT >= 6

FibreRequirement: 0.005 Ingr_BEEF + 0.001 Ingr_CHICKEN + 0.003 Ingr_MUTTON
 + 0.1 Ingr_RICE + 0.15 Ingr_WHEAT <= 2

SaltRequirement: 0.005 Ingr_BEEF + 0.002 Ingr_CHICKEN + 0.007 Ingr_MUTTON
 + 0.002 Ingr_RICE + 0.008 Ingr_WHEAT <= 0.4

VARIABLES
Ingr_BEEF Continuous
Ingr_CHICKEN Continuous
Ingr_GEL Continuous
Ingr_MUTTON Continuous
Ingr_RICE Continuous
Ingr_WHEAT Continuous

In [15]:
# The problem is solved using PuLP's choice of Solver
prob.solve()

1

In [17]:
# Each of the variables is printed with it's resolved optimum value
for v in prob.variables():
    print(v.name, "=", v.varValue)
# The optimised objective function value is printed to the screen
print("Total Cost the Cat Food = ", value(prob.objective))

Ingr_BEEF = 60.0
Ingr_CHICKEN = 0.0
Ingr_GEL = 40.0
Ingr_MUTTON = 0.0
Ingr_RICE = 0.0
Ingr_WHEAT = 0.0
Total Cost the Cat Food =  0.52


In [19]:
import pandas as pd
#Report Sensitivity Analysis
print("\nSensitivity Analysis")
Cons_Sensitivity_Report=[{'Constraint_Name':name,'Slack':c.slack,"Shadow_Price":c.pi}
                         for name, c in prob.constraints.items()]
print(pd.DataFrame(Cons_Sensitivity_Report))
print("\n")
Variable_Sensitivity_Report = [{'Variable_Name': v.name, 'Value':v.varValue,'Reduced_Cost': v.dj}
                               for v in prob.variables()]

print(pd.DataFrame(Variable_Sensitivity_Report)[['Variable_Name','Value','Reduced_Cost']])


Sensitivity Analysis
      Constraint_Name  Slack  Shadow_Price
0            GramsSum   -0.0         0.001
1  ProteinRequirement   -4.0         0.000
2      FatRequirement   -0.0         0.070
3    FibreRequirement    1.7         0.000
4     SaltRequirement    0.1         0.000


  Variable_Name  Value  Reduced_Cost
0     Ingr_BEEF   60.0        0.0000
1  Ingr_CHICKEN    0.0        0.0064
2      Ingr_GEL   40.0        0.0000
3   Ingr_MUTTON    0.0        0.0013
4     Ingr_RICE    0.0        0.0003
5    Ingr_WHEAT    0.0        0.0033
